## Graph Setup

#### Base

In [1]:
from typing_extensions import TypedDict

from langgraph.graph import StateGraph,START, END
from langgraph.graph.message import add_messages

import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from typing import Annotated
class State(TypedDict):
    messages:Annotated[list, add_messages]

In [3]:
from langchain.chat_models import init_chat_model

# llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
llm = init_chat_model("groq:qwen/qwen3.8-27b")
llm.profile

#### StateGraph

In [4]:
## Stategraph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

In [5]:
memory = MemorySaver()  # initalize the memory

builder = StateGraph(State)

In [6]:
def llm_call(state:State):
    return{ "messages": [llm.invoke(state["messages"])]}

In [7]:
builder.add_node("calling_llm", llm_call)

builder.add_edge(START, "calling_llm")
builder.add_edge("calling_llm",END)

graph = builder.compile(checkpointer=memory) # add the memory in graph

## Streaming

Methods: .stream() and astream()

- These methods are sync and async methods for streaming back results.

Additional parameters in streaming modes for graph state

- **values** : This streams the full state of the graph after each node is called.
- **updates** : This streams updates to the state of the graph after each node is called.

In [ ]:
config={"configurable":{"thread_id":"1"}}

# stream_mode='updates' # last updated message will be displayed

for chunk in graph.stream({"messages":"Hello, My name is Naveen and I like Chess"},config, stream_mode='updates'):
    print(chunk)

{'calling_llm': {'messages': [AIMessage(content="Hello Naveen! It's great to meet you. Chess is a wonderful hobby.\n\nSince you mentioned you like chess, how would you like to proceed? Here are a few things I can help you with:\n\n1. **Play a game**: We can play a text-based chess game right here.\n2. **Puzzles & Tactics**: I can give you tactical puzzles to solve.\n3. **Strategy & Theory**: Ask me about specific openings, endgame techniques, or famous games.\n4. **Analysis**: Share a position (using FEN or a board description) and I can offer suggestions.\n\nLet me know what interests you most! ♟️", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 180, 'total_tokens': 321, 'completion_time': 0.268522536, 'completion_tokens_details': None, 'prompt_time': 0.012437467, 'prompt_tokens_details': None, 'queue_time': 0.052364711, 'total_time': 0.280960003}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tie

In [ ]:
# stream_mode='values' # all messages will be displayed and added next messages in it

for chunk in graph.stream({"messages":"Hello, My name is Naveen and I like Chess"},config, stream_mode='values'):
    print(chunk)

{'messages': [HumanMessage(content='Hello, My name is Naveen and I like Chess', additional_kwargs={}, response_metadata={}, id='d8ca9226-0ded-4fa3-a3e7-6ea1ecf0a9bb'), AIMessage(content="Hello, Naveen! It's nice to meet you. Chess is a fascinating game with a rich history and deep strategic complexity.\n\nAre you a casual player, or do you play competitively? I'd love to hear about your experience. For example:\n*   Do you have a preferred opening?\n*   Do you enjoy playing online, over the board, or both?\n*   Are you looking to improve your tactical skills, endgame knowledge, or something else?\n\nI can also help with puzzles, explain rules, analyze positions, or just chat about chess history and famous players. What would you like to explore?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 24, 'total_tokens': 155, 'completion_time': 0.248421345, 'completion_tokens_details': None, 'prompt_time': 0.001429227, 'prompt_tokens_details

#### astream()

In [ ]:
config={"configurable":{"thread_id":"2"}}

## more detailed information will be available

async for event in graph.astream_events({"messages":"Hello, My name is Naveen and I like Chess"},config, version='v2'):
    print(event)

{'event': 'on_chain_start', 'data': {'input': {'messages': 'Hello, My name is Naveen and I like Chess'}}, 'name': 'LangGraph', 'tags': [], 'run_id': '01a15f6c-4052-711c-9d78-5755b2733ee3', 'metadata': {'thread_id': '2', 'ls_integration': 'langgraph'}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='Hello, My name is Naveen and I like Chess', additional_kwargs={}, response_metadata={}, id='fc925a17-854b-40de-9c12-1e6a27a434dd')]}}, 'name': 'calling_llm', 'tags': ['graph:step:1'], 'run_id': '01a15f6c-4054-7599-979d-05e0b37db960', 'metadata': {'thread_id': '2', 'ls_integration': 'langgraph', 'langgraph_step': 1, 'langgraph_node': 'calling_llm', 'langgraph_triggers': ('branch:to:calling_llm',), 'langgraph_path': ('__pregel_pull', 'calling_llm'), 'langgraph_checkpoint_ns': 'calling_llm:b4221ba1-4744-09db-b093-5d9f7e75a5a1'}, 'parent_ids': ['01a15f6c-4052-711c-9d78-5755b2733ee3']}
{'event': 'on_chat_model_start', 'data': {'input': {'message